In [12]:
import os
import subprocess
import weatherbench2
import ast
import apache_beam

from absl import app
from absl import flags
from weatherbench2 import config
from weatherbench2 import evaluation
from weatherbench2 import flag_utils
from weatherbench2 import metrics
from weatherbench2 import thresholds
from weatherbench2.derived_variables import DERIVED_VARIABLE_DICT
from weatherbench2.regions import CombinedRegion
from weatherbench2.regions import LandRegion
from weatherbench2.regions import SliceRegion
import xarray as xr

# Set your bucket, project, and region
BUCKET = 'my-bucket'
PROJECT = 'my-project'
REGION = 'us-central1'
output_dir = "/Users/ohouck/Library/CloudStorage/OneDrive-TheUniversityofChicago/ai_weather_ag/forecasts"

# Define the list of forecast paths and variables to evaluate
forecast_paths = [
    'gs://weatherbench2/datasets/hres/2016-2022-0012-64x32_equiangular_with_poles_conservative.zarr',
]

# variable_sets = [
#     ['geopotential', 'temperature', 'u_component_of_wind', 'v_component_of_wind'],
#     ['specific_humidity', '2m_temperature', 'mean_sea_level_pressure'],
# ]
variable_sets = ['geopotential']

# Set the common arguments for all evaluations
common_args = [
    f'--obs_path=gs://weatherbench2/datasets/era5/2016-2022-0012-64x32_equiangular_conservative.zarr',
    f'--climatology_path=gs://weatherbench2/datasets/era5-hourly-climatology/1990-2017_6h_64x32_equiangular_with_poles_conservative.zarr',
    f'--output_dir={output_dir}',
    '--input_chunks=time=1,lead_time=1',
    '--eval_configs=deterministic',
]


forecast_path = 'gs://weatherbench2/datasets/hres/2016-2022-0012-64x32_equiangular_conservative.zarr'
xr.open_zarr(forecast_path)


exit()

# Iterate over the forecast paths and variable sets
for forecast_path, variables in zip(forecast_paths, variable_sets):
    print(forecast_path, variables)
    # Build the command with the specific forecast path and variables
    cmd = [
        'python3', 'evaluate.py',
        f'--forecast_path={forecast_path}',
        f'--variables={",".join(variables)}',
    ] + common_args 

    # Run the evaluation command
    subprocess.run(cmd, check=True)

# # Iterate over the forecast paths and variable sets
# for forecast_path, variables in zip(forecast_paths, variable_sets):
#     # Build the command with the specific forecast path and variables
#     cmd = [
#         'python3', 'evaluate.py',
#         f'--forecast_path={forecast_path}',
#         f'--variables={",".join(variables)}',
#     ] + common_args + [
#         '--',
#         f'--project={PROJECT}',
#         f'--region={REGION}',
#         f'--temp_location=~/Downloads',
#         '--setup_file=setup.py',
#         '--requirements_file=requirements.txt',
#         '--job_name=eval-$USER',
#     ]

#     # Run the evaluation command
#     subprocess.run(cmd, check=True)

FileNotFoundError: No such file or directory: 'gs://weatherbench2/datasets/hres/2016-2022-0012-64x32_equiangular_with_poles_conservative.zarr'

In [ ]:
# Pip might complain about the Pandas version. The notebook should still work as expected.
# !pip install git+https://github.com/google-research/weatherbench2.git


import apache_beam   # Needs to be imported separately to avoid TypingError
import weatherbench2
from weatherbench2 import config
from weatherbench2.metrics import MSE, ACC
from weatherbench2.regions import SliceRegion
from weatherbench2.evaluation import evaluate_in_memory
import xarray as xr


# Run the code below to access cloud data on Colab!
# from google.colab import auth
# auth.authenticate_user()

# pangu_path = 'gs://weatherbench2/datasets/pangu/2018-2022_0012_64x32_equiangular_conservative.zarr'
# gcm_deterministic_path = 'gs://weatherbench2/datasets/neuralgcm_deterministic/2020-64x32_equiangular_conservative.zarr'
# ifs_mean_path = 'gs://weatherbench2/datasets/ifs_ens/2018-2022-64x32_equiangular_conservative_mean.zarr'
# obs_path = 'gs://weatherbench2/datasets/era5/1959-2022-6h-64x32_equiangular_conservative.zarr'
# climatology_path = 'gs://weatherbench2/datasets/era5-hourly-climatology/1990-2019_6h_64x32_equiangular_conservative.zarr'
# output_path = "/Users/ohouck/Library/CloudStorage/OneDrive-TheUniversityofChicago/ai_weather_ag/forecasts"


# #bbox are [lon_min, lat_min, lon_max, lat_max]
# bbox_global = [-180, -90, 180, 90]
# bbox_midwest = [-100, 35, -80, 50]
# bbox_pakistan = [60, 24, 78, 37]


# # paths = config.Paths(
# #     forecast = {
# #         'pangu': pangu_path,
# #         'neuralgcm_deterministic': gcm_deterministic_path,
# #         'ifs_mean': ifs_mean_path,
# #     },
# #     obs = obs_path,
# #     output_dir = f'{output_path}/'
# # )

# pangu_path = config.Paths(
#     forecast = pangu_path,
#     obs = obs_path,
#     output_dir = f'{output_path}/'
# )

# selection = config.Selection(
#     variables = [
#         '2m_temperature',
#     ], 
#     time_slice=slice('2020-03-01', '2020-04-01'),
# )

# pangu_config = config.Data(selection=selection, paths=pangu_path)

# regions = {
#     'global': SliceRegion(),
#     'pakistan': SliceRegion(lat_slice = slice(bbox_pakistan[1], bbox_pakistan[3]), 
#                             lon_slice = slice(bbox_pakistan[0], bbox_pakistan[2])),
# }

# climatology = xr.open_zarr(climatology_path)

# eval_config = { 
#     'deterministic': config.Eval(
#         metrics = {
#             'MSE': MSE(),
#             'ACC': ACC(climatology = climatology)
#             },
#         regions = regions,
#     )
# }

def eval_forecast(variables, time_start, time_end, forecast_path, obs_path, 
                  climatology_path, regions, output_path, output_name):
    path = config.Paths(
        forecast = forecast_path,
        obs = obs_path,
        output_dir = f'{output_path}/'
    )

    selection = config.Selection(
        variables = variables,
        time_slice=slice(time_start, time_end),
    )

    config_data = config.Data(selection=selection, paths=path)

    eval_config = {
        output_name: config.Eval(
            metrics = {
                'MSE': MSE(),
                'ACC': ACC(climatology = climatology_path)
                },
            regions = regions,
        )
    }

    evaluate_in_memory(config_data, eval_config)



In [8]:

import apache_beam   # Needs to be imported separately to avoid TypingError
import weatherbench2
from weatherbench2 import config
from weatherbench2.metrics import MSE, ACC
from weatherbench2.regions import SliceRegion
from weatherbench2.evaluation import evaluate_in_memory
import xarray as xr

def eval_forecast(variables, time_start, time_end, forecast_path, obs_path, 
                  climatology_path, regions, output_path, output_name):
    path = config.Paths(
        forecast = forecast_path,
        obs = obs_path,
        output_dir = f'{output_path}/'
    )

    selection = config.Selection(
        variables = variables,
        time_slice=slice(time_start, time_end),
    )

    config_data = config.Data(selection=selection, paths=path)
    climatology = xr.open_zarr(climatology_path)

    eval_config = {
        output_name: config.Eval(
            metrics = {
                'MSE': MSE(),
                'ACC': ACC(climatology = climatology)
                },
            regions = regions,
        )
    }

    evaluate_in_memory(config_data, eval_config)

#bbox are [lon_min, lat_min, lon_max, lat_max]
bbox_midwest = [-100, 35, -80, 50]
bbox_pakistan = [60, 24, 78, 37]

regions = {
    'global': SliceRegion(),
    'pakistan': SliceRegion(lat_slice = slice(bbox_pakistan[1], bbox_pakistan[3]), 
                            lon_slice = slice(bbox_pakistan[0], bbox_pakistan[2])),
}

variables = ['temperature']
levels = ['500, 700, 850']
time_start = '2020-04-01'
time_end = '2020-04-10'
pangu_path = 'gs://weatherbench2/datasets/pangu/2018-2022_0012_64x32_equiangular_conservative.zarr'
gcm_deterministic_path = 'gs://weatherbench2/datasets/neuralgcm_deterministic/2020-64x32_equiangular_conservative.zarr'
ifs_mean_path = 'gs://weatherbench2/datasets/ifs_ens/2018-2022-64x32_equiangular_conservative_mean.zarr'
obs_path = 'gs://weatherbench2/datasets/era5/1959-2022-6h-64x32_equiangular_conservative.zarr'
climatology_path = 'gs://weatherbench2/datasets/era5-hourly-climatology/1990-2019_6h_64x32_equiangular_conservative.zarr'
output_path = "/Users/ohouck/Library/CloudStorage/OneDrive-TheUniversityofChicago/ai_weather_ag/forecasts"

# pangu
eval_forecast(variables = variables, time_start = time_start, time_end = time_end, 
              forecast_path = pangu_path, obs_path = obs_path, climatology_path = climatology_path, 
              regions = regions, output_path = output_path, output_name = 'pangu_2m_weatherbench')

# # gcm_deterministic
# eval_forecast(variables = variables, time_start = time_start, time_end = time_end, 
#               forecast_path = gcm_deterministic_path, obs_path = obs_path, climatology_path = climatology_path, 
#               regions = regions, output_path = output_path, output_name = 'gcm_deterministic_2m_weatherbench')
# # ifs_mean
# eval_forecast(variables = variables, time_start = time_start, time_end = time_end, 
#               forecast_path = ifs_mean_path, obs_path = obs_path, climatology_path = climatology_path, 
#               regions = regions, output_path = output_path, output_name = 'ifs_mean_2m_weatherbench')

In [9]:
results = xr.open_dataset(f'{output_path}/pangu_2m_weatherbench.nc')
print(results)

print(results.lead_time.values[1])

results = xr.concat(
    [results,
     results.sel(metric=['MSE']).assign_coords(metric = ['RMSE']) ** 0.5
     ],
    dim='metric'
)


# plot rmse for each region
results['2m_temperature'].sel(metric='RMSE').plot(col='region', col_wrap=2)

<xarray.Dataset>
Dimensions:      (lead_time: 40, level: 13, region: 2, metric: 2)
Coordinates:
  * lead_time    (lead_time) timedelta64[ns] 0 days 06:00:00 ... 10 days 00:0...
  * level        (level) int32 1000 925 850 700 600 500 ... 250 200 150 100 50
  * region       (region) object 'global' 'pakistan'
  * metric       (metric) object 'ACC' 'MSE'
Data variables:
    temperature  (metric, region, lead_time, level) float64 ...
43200000000000 nanoseconds


KeyError: '2m_temperature'